In [1]:
from pathlib import Path

import geopandas as gpd
import requests
from tqdm.notebook import tqdm

In [2]:
S3_URL_PREFIX = "https://esa-worldcover.s3.eu-central-1.amazonaws.com"

# Download WorldCover tiles (global land cover data)

## 1. User input

In [3]:
year = 2021  # setting this to 2020 will download the v100 product instead
output_folder = r"/home/jovyan/shared/disk/01_datasets/worldcover"
countries = ["Norway", "Sweden", "Finland"]

## 2. Identify tiles

In [4]:
# Load Natural Earth data
ne_url = (
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)
ne = gpd.read_file(ne_url)

# Filter to countries of interest
selected = ne[ne["NAME"].isin(countries)]
geom = selected.geometry.union_all()

# Load WorldCover grid
grid_url = f"{S3_URL_PREFIX}/esa_worldcover_grid.geojson"
grid = gpd.read_file(grid_url)

# Get intersecting tiles
tiles = grid[grid.intersects(geom)]

print(len(tiles), "tiles within the area of interest.")

tiles.head()

54 tiles within the area of interest.


,ll_tile,geometry
1131,N66E030,"POLYGON ((30 66, 30 69, 33 69, 33 66, 30 66))"
1132,N66E027,"POLYGON ((27 66, 27 69, 30 69, 30 66, 27 66))"
1133,N66E024,"POLYGON ((24 66, 24 69, 27 69, 27 66, 24 66))"
1134,N69E024,"POLYGON ((24 69, 24 72, 27 72, 27 69, 24 69))"
1135,N69E027,"POLYGON ((27 69, 27 72, 30 72, 30 69, 27 69))"


## 3. Download tiles

In [5]:
# Get data
version = {2020: "v100", 2021: "v200"}[year]
for tile in tqdm(tiles.ll_tile):
    url = f"{S3_URL_PREFIX}/{version}/{year}/map/ESA_WorldCover_10m_{year}_{version}_{tile}_Map.tif"
    r = requests.get(url, allow_redirects=True)
    out_fn = Path(output_folder) / Path(version) / Path(url).name
    out_fn.parent.mkdir(parents=True, exist_ok=True)
    with open(out_fn, "wb") as f:
        f.write(r.content)

  0%|          | 0/54 [00:00<?, ?it/s]

## 4. Merge tiles